In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")
from modeling import COVARIATES, N_PERMUTATIONS, bh_fdr, oof_pearson, permutation_p, permuted

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
TRAIN_DIR = DATA_DIR / "train_diversity"
TARGETS = ["Richness", "Shannon_diversity"]

model_input = pd.read_csv(TRAIN_DIR / "model_input_diet_oral_diversity_baseline.csv", low_memory=False)
results = pd.read_csv(TRAIN_DIR / "model_results_diet_predict_oral_diversity_baseline.csv")
features = COVARIATES + [c for c in model_input.columns if c not in ["participant_id", *COVARIATES, *TARGETS]]
observed = results[results["model"] == "covariates_plus_diet"].set_index("target")["pearson_r"]

In [ ]:
null = {}
for target in TARGETS:
    y = model_input[target].to_numpy(dtype=float)
    null[target] = [oof_pearson(model_input[features], permuted(y, seed)) for seed in range(N_PERMUTATIONS)]

permutation = pd.DataFrame({
    "target": TARGETS,
    "observed_pearson_r": [observed[t] for t in TARGETS],
    "perm_p_value": [permutation_p(null[t], observed[t]) for t in TARGETS],
})
permutation["perm_q_value"] = bh_fdr(permutation["perm_p_value"])
permutation["perm_significant_fdr"] = permutation["perm_q_value"] < 0.05

out_dir = TRAIN_DIR / "alpha_diversity_permutation_test"
out_dir.mkdir(parents=True, exist_ok=True)
permutation.to_csv(out_dir / "alpha_diversity_permutation_results_baseline.csv", index=False)
pd.DataFrame(null).rename_axis("permutation").to_csv(out_dir / "alpha_diversity_permutation_null_scores_baseline.csv")
print(permutation)